## Figure 1 Generation

### The goal of this notebook is to produce a plot for Figure 1. 

General Schema:

For a given climate variable, say, tas, plot its global average value over time from 2015 to 2100, 1 thin line per each ensemble per model and one thick line per average of all models, for both a non net-zero scenario and a net-zero scenario.

Indicate approximate years of net-zero accomplishment in both scenarios.

Label accordingly.

### Imports and File Stitching

In [ ]:
# Imports
import os
import re
import numpy as np
import netCDF4 as nc
from netCDF4 import Dataset
import matplotlib.pyplot as plt

# ----File Stitching----
# If in prep folder, cd back to base repository folder
if os.path.basename(os.getcwd()) == "prep":
    os.chdir('../..')

# Control which user's files are accessed
match_sk = 'sophiekim'
match_hc = 'hayeonchung'
match_ck = 'Caroline'
match_st = 'student'
path_str = os.getcwd()
print(path_str)
if re.search(match_sk, path_str):
    os.chdir("/Users/sophiekim/Desktop/2_research/MamalakisResearch") 
    user = match_sk
    print("Sophie Kim recognized as user.")
elif re.search(match_hc, path_str):
    os.chdir("/Users/hayeonchung/Downloads/Mamalakis Graduate Research/MamalakisResearch") 
    user = match_hc
    print("Hayeon Chung recognized as user.")
elif re.search(match_ck, path_str):
    os.chdir("/Users/Caroline/Desktop/school/MamalakisResearch") 
    user = 'carolinekranefuss'
    print("Caroline Kranefuss recognized as user.")
elif re.search(match_st, path_str):
    os.chdir("/Users/student/Desktop/mamalakis_research/MamalakisResearch")
    user = match_st
    print("Student recognized as user.")
else:
    print("User not recognized. Please manually change directory.")

In [ ]:
# Assign base path
base_path = os.getcwd()

# All users should have locally loaded 'data' folder
data_path = base_path + '/data/'

In [ ]:
# initializing model list variable to call in function
model_list = [
    "CNRM_ESM2-1_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc",
    "MIROC6_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc",
    "MPI-ESM1-2-LR_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc",
    "MRI-ESM2-0_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc",
    "UKESM1-0-LL_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc",
]

model_paths = [data_path + model for model in model_list]

model_paths

### Sub Functions

In [ ]:
def convert_units(varname: str, x: np.ndarray):
    """
    varname: index number from the list of variables so get_data func can convert units 
    x: data that needs units changed (raw x data) in get_data func 
    """
    if varname in {"tas", "tasmax", "tasmin"}:
        # kelvin to celsius
        return x - 273.15, "$^{\circ}$C"
    if varname == "pr":
        # kg/(m2*s) to mm/day
            # 1kg/m2 = 1 mm 
        return x * 86400.0, "mm/day"
    if varname == "psl":
        # pascals to hpa
        return x / 100.0, "hPa"
    
    # these don't need to be converted -- just adding the units 
    if varname == "sfcWind":
        return x, "m/s"
    if varname == "mrsos":
        return x, "kg/m$^{2}$"
    return x, "unknown"

In [ ]:
# Define global variable for script
var = 'tas'

In [ ]:
def var_yr_avgs(model, scenario, var=var):
    """
    A function to obtain the average yearly global value of a variable (year by year) for a given scenario and model for each ensemble of that model
    Default is ssp119
    Returns: An array of floats
    """

    # Define order of variables in data loaded to obtain index
    var_list = ["tas", "tasmax", "tasmin", "pr", "psl", "sfcWind", "mrsos"]
    for i, item in enumerate(var_list):
        if re.match(item, var):
            var_index = i

    # Months range from 1 to 1032 but numpy is zero-indexed, so 0 to 1031 
    months = np.arange(0,1032,1)   

    # Make an array for each ensemble, 5 ensembles per model 
    ensembles = [[],[],[],[],[]]

    # loading the data for specific model 
    with nc.Dataset(model) as ds:

        # For each of the 5 ensembles
        for j in range(5):
            
            # Take the average of all global values for the variable per month and append to the appropriate ensembles array
            for month in months:

                # Slice dimensions for the ensemble, the specific var index and all the lon/lat dimensions for EACH month; then find average of all global values for that month
                # All global point values for that month/variable/ensemble combo
                vals_mth_ensemble = ds[f"data_{scenario}"][j, var_index, month, :, :]  # (144, 73) -> 10,512 values

                # Scale up/down depending on location
                lats = ds['lat'] # (73,)
                lons = ds['lon']
                cos_lats = np.cos(np.array(lats) * np.pi / 180)
                # Broadcast cos of lats 144 times
                cos_lats_144 = np.tile(cos_lats, (len(lons), 1)) # (144, 73)
                # Hadamard product --> monthly variable values weighted by their location
                vals_cos = vals_mth_ensemble * cos_lats_144

                # Take the mean of those 10,512 weighted values
                global_avg_ens_mth = np.nanmean(vals_cos, dtype=np.float64) # Single number

                ensembles[j].append(global_avg_ens_mth)
    # Now we have an list of lists - average global variable values for every month, repeated across each ensemble of a model - each list has 1032 values and there are 5 lists (ensembles)

    # Convert to yearly averages
    ensembles_yr = [] 
    for ensemble in ensembles:
        # Convert to array 
        ensemble_arr = np.array(ensemble)
        # Reshape
        reshaped = ensemble_arr.reshape((len(ensemble_arr)//12, 12)) #(86, 12) aka num years, 12 months per year
        yearly_ens = np.mean(reshaped, axis=1, dtype=np.float64) # Shape (86,) to collapse axis 1 (months)
        # Do unit conversion 
        units_changed, _ = convert_units(var, yearly_ens)
        # Assign
        ensembles_yr.append(units_changed)

    return ensembles_yr

print(len(var_yr_avgs(model_paths[0], scenario='ssp119')[0])) # Sanity check: Confirmed to contain 86 years

print(var_yr_avgs(model_paths[0], 'ssp126')) # Sanity check: does ~16 degrees Celsius make sense for a January temperature? Calculated only over land 
                                             # There are 2 poles (cold) but only one equator (hot) so those would drag the temp down
                                                # Temp should be upscaled in tropics, though, because the area is physically greater 
                                             # ~60 degrees Fahrenheit - makes sense 

**meeting notes 06/01:**

- each point = global annual avg per variable for each trajectory within the model 
    - so 5 time series and using standardized values from the trajectories 
- standardize by getting avg of baseline time period (should be 50 pts) and then subtract the baseline from each point 
    - (global annual avg - mean from 2015-2024) / std from 2015-2024
    - do this separately for each model --> bc each model might have diff avgs and stds 
        - and separately for each scenario 

- standardization should be done under each trajectory and under each model 


simplified version of time series for figure 1A 

I need to combine each ensemble into a single model at the very end or not at all and do calculations incl std FIRST

In [ ]:
def baseline_std_adj(model_var_over_time):
    """
    A function to convert raw temperatures to standardized deviations in a single model
    Output: Array of floats
    """

    # Baseline is the average of the first decade
    ens_bases = []
    for ensemble in model_var_over_time:
        ens_base = ensemble[0:10]
        ens_bases.append(ens_base) # 50 item-long list
    baseline = np.mean(ens_bases, axis=(0,1), dtype=np.float64)

    # Calculate the standard deviation of the baseline
    std = np.std(ens_bases, axis=(0,1), dtype=np.float64)
    std = 1

    diff_over_time = model_var_over_time - baseline
    final = diff_over_time / std

    return final

### Main Time Series Plot Function

In [ ]:
def time_series_plot(model_paths, ssp119='yes', ssp126='yes', year_lines='yes'):

    # -------------- FORMAT AND SETUP -----------------------

    # Create a range for the x-axis (time)
    yrs = np.arange(2015, 2101, 1) 
    
    # Check variables and obtain y-axis label
    if var == 'tas':
        y = 'Mean Surface Temperature ($^{\circ}$C)'
    elif var == 'tasmax':
        y = 'Max Surface Temperature ($^{\circ}$C)'
    elif var == 'tasmin':
        y = 'Min Surface Temperature ($^{\circ}$C)'
    elif var == 'pr':
        y = 'Precipitation (mm/day)'
    elif var == 'psl':
        y = 'Pressure at Sea Level (hPa)'
    elif var == 'sfcWind':
        y = 'Surface Wind Speed (m/s)'
    elif var == 'mrsos':
        y = 'Moisture in Upper Soil Column (kg/m$^{2}$)'
    else:
        print('Variable must be tas, tasmax, tasmin, pr, psl, sfcWind, or mrsos.')
        return

    # Set up plot 
    fig, ax = plt.subplots(figsize=(20, 6))
    plt.locator_params(axis='x', nbins=10)

    # Create a box for baseline climate
    ax.axvspan(2015, 2024, color='grey', alpha=0.2, zorder=0, label='Baseline Climate (2015-2024)')
    
    # ------------ SSP 119 PLOTTING--------------------------
    
    # Run yearly averages and conversion functions for ssp119 (includes unit and diff/std conversions)
    results_119 = []
    if ssp119 == 'yes':

        # ----- SUB FUNCTIONS ----------
        for model in model_paths:
            yr_avgs = var_yr_avgs(model, 'ssp119') # A list of 5 lists (each 86 values long) aka 5 ensembles per model
            adj = baseline_std_adj(yr_avgs) # Still same dimensions as above
            results_119.append(adj) # Adding to all results
            
        # ----- AVERAGING -----------
        # Get average of each model's ensembles
        model_averages_119 = []
        # For every model (for every list of lists)  
        for model in results_119:
            model_avg = [np.mean(ensemble, dtype=np.float64) for ensemble in zip(*model)] 
            model_averages_119.append(model_avg)
        # Get average of models
        average_119 = [np.mean(model, dtype=np.float64) for model in zip(*model_averages_119)] 

        # ----- PLOTTING -----------
        # Each ensemble is a list of 5 lists of 86; there are 5 models, so 25 lines to plot
        # For every model in the results
        for model in results_119:
            # For every ensemble in every model
            for ensemble in model:
                # Gives 25 lines
                ax.plot(yrs, ensemble, color='#8cc5e3', linewidth=1, alpha=0.5)
        # Dummy data for single label
        ax.plot([], [], label='SSP1-1.9 Model Ensembles', color='#8cc5e3', linewidth=1, alpha=0.5)


    # ------------ SSP 126 PLOTTING --------------------------

    results_126 = []
    if ssp126 == 'yes':

        # ----- SUB FUNCTIONS ----------
        for model in model_paths:
            yr_avgs = var_yr_avgs(model, 'ssp126')
            adj = baseline_std_adj(yr_avgs)
            results_126.append(adj)
        
         # ----- AVERAGING -----------
        model_averages_126 = []
        for model in results_126:
            model_avg = [np.mean(ensemble, dtype=np.float64) for ensemble in zip(*model)] 
            model_averages_126.append(model_avg)
        average_126 = [np.mean(model, dtype=np.float64) for model in zip(*model_averages_126)] 
   
        # ----- PLOTTING -----------
        for model in results_126:
            for ensemble in model:
                plt.plot(yrs, ensemble, color='#f55f74', linewidth=1, alpha=0.5)
        ax.plot([], [], label='SSP1-2.6 Model Ensembles', color='#f55f74', linewidth=1, alpha=0.5)


    # ------MODEL AVERAGE LINES PLOTTING -----------

    # Plot model averages on top of other lines (ie last)
    if ssp119 == 'yes':
        ax.plot(yrs, average_119, label='Average: All Models in SSP1-1.9', color='#8cc5e3', linewidth=2)
    if ssp126 == 'yes':
        ax.plot(yrs, average_126, label='Average: All Models in SSP1-2.6', color='#f55f74', linewidth=2)


    # ----------------------- NET-ZERO LINES ------------------

    # Plot net-zero lines
    if year_lines == 'yes':

        # Find the index for the years 2050 and 2070 and 2026 in the 'yrs' array
        idx_2050 = np.where(yrs == 2050)[0][0]
        idx_2070 = np.where(yrs == 2070)[0][0]
        idx_2026 = np.where(yrs == 2026)[0][0]
        
        # Extract the exact y-values at those years from the average arrays
        # Plot current year at mean between ssp119 and ssp126 heights or at ssp119 height if no ssp126 (or 126/119)
        if ssp119 == 'yes' and ssp126 == 'yes':

            y_2050 = average_119[idx_2050]
            y_2070 = average_126[idx_2070]
            y_2026 = (average_119[idx_2026] + average_126[idx_2026])/2

            # Current y-axis limits (lines start at bottom)
            ymin, ymax = ax.get_ylim()

            plt.vlines(x=2050, ymin=ymin, ymax=y_2050, color='#8cc5e3', linestyle = 'dashed', label='Net Zero ~2050 (SSP1-1.9)')
            plt.vlines(x=2070, ymin=ymin, ymax=y_2070, color='#f55f74', linestyle = 'dashed', label='Net Zero ~2070 (SSP1-2.6)')
            plt.vlines(x=2026, ymin=ymin, ymax=y_2026, color="#AAAAAA", linestyle = 'dashed', label='Current Year')

        if ssp119 == 'no':

            y_2070 = average_126[idx_2070]
            y_2026 = average_126[idx_2026]

            # Current y-axis limits (lines start at bottom)
            ymin, ymax = ax.get_ylim()

            plt.vlines(x=2070, ymin=ymin, ymax=y_2070, color='#f55f74', linestyle = 'dashed', label='Net Zero ~2070 (SSP1-2.6)')
            plt.vlines(x=2026, ymin=ymin, ymax=y_2026, color="#AAAAAA", linestyle = 'dashed', label='Current Year')

        if ssp126 == 'no':

            y_2050 = average_119[idx_2050]
            y_2026 = average_119[idx_2026]

            # Current y-axis limits (lines start at bottom)
            ymin, ymax = ax.get_ylim()

            plt.vlines(x=2050, ymin=ymin, ymax=y_2050, color='#8cc5e3', linestyle = 'dashed', label='Net Zero ~2050 (SSP1-1.9)')
            plt.vlines(x=2026, ymin=ymin, ymax=y_2026, color="#AAAAAA", linestyle = 'dashed', label='Current Year')

        
        # Re-set ymin, ymax to not extend bottom of graph
        ax.set_ylim(ymin, ymax)


    # ------------ FORMATTING ---------
    # Formatting
    plt.xlabel("Year")
    plt.ylabel(f"{y} Deviation (Global Annual Mean)")
    plt.legend()
    plt.title(f"{y} Deviation, Global Annual Mean 2015 to 2100")
     # Move legend
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.)
    # Make room for the right-side legend
    plt.tight_layout()
    plt.subplots_adjust(right=0.5)
    
    # ---------- FINAL PLOTTING COMMAND -------------
    plt.show()


# --------------- RUNNING FUNCTION -----------
# With year lines
time_series_plot(model_paths)
time_series_plot(model_paths, ssp126='no')

# Without year lines
time_series_plot(model_paths, ssp126='yes', year_lines='no')
time_series_plot(model_paths, ssp126='no', year_lines='no')

In [ ]:
# Reset global variable
var = 'tasmin'

# Plot
time_series_plot(model_paths)
time_series_plot(model_paths, ssp126='no')
time_series_plot(model_paths, ssp126='yes', year_lines='no')
time_series_plot(model_paths, ssp126='no', year_lines='no')

In [ ]:
# Reset global variable
var = 'tasmax'

# Plot
time_series_plot(model_paths)
time_series_plot(model_paths, ssp126='no')
time_series_plot(model_paths, ssp126='yes', year_lines='no')
time_series_plot(model_paths, ssp126='no', year_lines='no')

In [ ]:
# Reset global variable
var = 'pr'

# Plot
time_series_plot(model_paths)
time_series_plot(model_paths, ssp126='no')
time_series_plot(model_paths, ssp126='yes', year_lines='no')
time_series_plot(model_paths, ssp126='no', year_lines='no')

In [ ]:
# Reset global variable
var = 'psl'

# Plot
time_series_plot(model_paths)
time_series_plot(model_paths, ssp126='no')
time_series_plot(model_paths, ssp126='yes', year_lines='no')
time_series_plot(model_paths, ssp126='no', year_lines='no')

In [ ]:
# Reset global variable
var = 'sfcWind'

# Plot
time_series_plot(model_paths)
time_series_plot(model_paths, ssp126='no')
time_series_plot(model_paths, ssp126='yes', year_lines='no')
time_series_plot(model_paths, ssp126='no', year_lines='no')

In [ ]:
# Reset global variable
var = 'mrsos'

# Plot
time_series_plot(model_paths)
time_series_plot(model_paths, ssp126='no')
time_series_plot(model_paths, ssp126='yes', year_lines='no')
time_series_plot(model_paths, ssp126='no', year_lines='no')